# SE4050 Deep Learning 2026 — Multi-Class Food Classification
## Member 3 (Kanushka) — MobileNetV2 Benchmark & Edge Efficiency Profiling

- **Module:** SE4050 – Deep Learning (2026)
- **Institution:** Sri Lanka Institute of Information Technology (SLIIT)
- **Assigned Model:** **MobileNetV2** (Lightweight Inverted Residual CNN)
- **Dataset:** Food-101 (101,000 images, 101 classes, ETH Zürich)
- **Working Branch:** `feature/kanushka-mobilenetv2`

---

### Core Research Objectives & Hypotheses for MobileNetV2:
1. **Depthwise Separable Convolutions:** By factoring standard 3×3 convolutions into separate spatial depthwise and channel pointwise convolutions, MobileNetV2 reduces computation by ~8× to 9×.
2. **Inverted Residuals & Linear Bottlenecks:** Information flows through low-dimensional bottleneck layers without non-linear ReLU destruction, maximizing representation capacity.
3. **Edge Deployment Suitability:** With only ~2.39M parameters (vs. ResNet50's ~25.6M), MobileNetV2 aims to demonstrate minimal degradation in top-1 accuracy while delivering significantly lower latency and disk footprint suitable for mobile nutrition-tracking apps.

In [1]:
# ==============================================================================
# Step 1: Environment Detection & Branch Synchronization
# ==============================================================================
import os
import sys

REPO_NAME = "food-classification-deep-learning"
REPO_URL = "https://github.com/niRmana11/food-classification-deep-learning.git"
BRANCH = "feature/kanushka-mobilenetv2"

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[INFO] Executing in Google Colab environment.")
    if not os.path.exists(f"/content/{REPO_NAME}"):
        print(f"[INFO] Cloning branch '{BRANCH}' from {REPO_URL}...")
        !git clone -b {BRANCH} {REPO_URL}
        %cd /content/{REPO_NAME}
    else:
        print(f"[INFO] Repository present. Pulling latest commits from '{BRANCH}'...")
        %cd /content/{REPO_NAME}
        !git pull origin {BRANCH}

    if f"/content/{REPO_NAME}" not in sys.path:
        sys.path.insert(0, f"/content/{REPO_NAME}")

    # GPU Verification
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"[SUCCESS] GPU Detected: {gpus[0].name}")
        !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
    else:
        print("[WARNING] No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")
else:
    print("[INFO] Executing in local environment.")
    import tensorflow as tf
    print("TensorFlow Version:", tf.__version__)


[INFO] Executing in Google Colab environment.
[INFO] Cloning branch 'feature/kanushka-mobilenetv2' from https://github.com/niRmana11/food-classification-deep-learning.git...
Cloning into 'food-classification-deep-learning'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 175 (delta 75), reused 134 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 21.02 MiB | 32.08 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/food-classification-deep-learning
[SUCCESS] GPU Detected: /physical_device:GPU:0
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14910 MiB


In [2]:
# ==============================================================================
# Step 2: Download & Extract Food-101 Dataset (~5GB)
# ==============================================================================
from src.data.download_food101 import download_and_extract_food101

# Automated download and extraction (skips if already present)
food101_dir = download_and_extract_food101(destination_dir="data/raw")
print(f"[INFO] Dataset ready at: {food101_dir}")


[INFO] Downloading Food-101 (~5GB) from http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz ...
[NOTE] On Google Colab, this takes ~1-2 minutes over cloud connection.


food-101.tar.gz: 5.00GB [02:32, 32.7MB/s]                            


[INFO] Download completed successfully.
[INFO] Extracting archive to /content/food-classification-deep-learning/data/raw ...
[INFO] Extraction completed.
[INFO] Dataset ready at: data/raw/food-101


In [3]:
# ==============================================================================
# Step 3: Load the Shared Multithreaded Preprocessing Pipeline
# ==============================================================================
from src.preprocessing.data_loader import get_food101_datasets

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

print("[INFO] Initializing tf.data pipelines for MobileNetV2 (normalization [-1, 1])...")
train_ds, val_ds, test_ds = get_food101_datasets(
    data_dir="data/raw/food-101",
    splits_dir="data/splits",
    model_type="mobilenetv2",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

# Inspect first batch
for images, labels in train_ds.take(1):
    print(f"[CHECK] Batch images shape: {images.shape}")
    print(f"[CHECK] Batch labels shape: {labels.shape}")
    print(f"[CHECK] Pixel range: min={tf.reduce_min(images):.2f}, max={tf.reduce_max(images):.2f}")
    break


[INFO] Initializing tf.data pipelines for MobileNetV2 (normalization [-1, 1])...
[CHECK] Batch images shape: (32, 224, 224, 3)
[CHECK] Batch labels shape: (32,)
[CHECK] Pixel range: min=-1.00, max=1.00


In [ ]:
# ==============================================================================
# Step 4: Build MobileNetV2 Architecture (Phase 1: Feature Extraction)
# ==============================================================================
from src.models.mobilenetv2 import build_mobilenetv2_model, get_model_parameter_stats

print("[INFO] Instantiating MobileNetV2 with frozen ImageNet backbone...")
model = build_mobilenetv2_model(
    num_classes=101,
    input_shape=(224, 224, 3),
    dropout_rate=0.2,
    trainable_base=False,
    learning_rate=0.001
)

stats = get_model_parameter_stats(model)
print("\n--- MobileNetV2 Parameter Footprint ---")
for k, v in stats.items():
    print(f"{k:30s}: {v}")

# Export model summary text for standardized results artifact
os.makedirs("results/mobilenetv2", exist_ok=True)
with open("results/mobilenetv2/model_summary.txt", "w") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))
print("\n[INFO] Saved results/mobilenetv2/model_summary.txt")


[INFO] Instantiating MobileNetV2 with frozen ImageNet backbone...

--- MobileNetV2 Parameter Footprint ---
model_name                    : MobileNetV2
total_parameters              : 2387365
trainable_parameters          : 129381
non_trainable_parameters      : 2257984
estimated_weights_size_mb     : 9.11



[INFO] Saved results/mobilenetv2/model_summary.txt


In [ ]:
# ==============================================================================
# Step 5: Train Phase 1 (Classifier Head Warmup)
# ==============================================================================
import time
import pandas as pd
from tensorflow.keras import callbacks

PHASE1_EPOCHS = 8

cb_phase1 = [
    callbacks.ModelCheckpoint(
        'results/mobilenetv2/best_phase1.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

print(f'[INFO] Starting Phase 1 training for {PHASE1_EPOCHS} epochs...')
start_time_phase1 = time.time()
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=cb_phase1
)
duration_phase1 = time.time() - start_time_phase1
print(f'[INFO] Phase 1 finished in {duration_phase1:.1f}s ({duration_phase1/60:.1f} min).')

# Immediately save Phase 1 history to disk so progress is never lost
h1_df = pd.DataFrame(history_phase1.history)
h1_df.index.name = 'epoch'
h1_df.to_csv('results/mobilenetv2/history_phase1.csv')
print('[INFO] Saved results/mobilenetv2/history_phase1.csv')


[INFO] Starting Phase 1 training for 8 epochs...
Epoch 1/8
2131/2131 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.3442 - loss: 2.7928 - top_5_accuracy: 0.5992
Epoch 1: val_loss improved from None to 1.91067, saving model to results/mobilenetv2/best_phase1.keras

Epoch 1: finished saving model to results/mobilenetv2/best_phase1.keras
2131/2131 ━━━━━━━━━━━━━━━━━━━━ 726s 332ms/step - accuracy: 0.4320 - loss: 2.3323 - top_5_accuracy: 0.7025 - val_accuracy: 0.5178 - val_loss: 1.9107 - val_top_5_accuracy: 0.7875 - learning_rate: 0.0010
Epoch 2/8
2131/2131 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - accuracy: 0.5210 - loss: 1.9134 - top_5_accuracy: 0.7853
Epoch 2: val_loss improved from 1.91067 to 1.82823, saving model to results/mobilenetv2/best_phase1.keras

Epoch 2: finished saving model to results/mobilenetv2/best_phase1.keras
2131/2131 ━━━━━━━━━━━━━━━━━━━━ 685s 321ms/step - accuracy: 0.5238 - loss: 1.9050 - top_5_accuracy: 0.7869 - val_accuracy: 0.5443 - val_loss: 1.8282 - val_top_5_accurac

In [ ]:
# ==============================================================================
# STEP 5B (RECOMMENDED): Save Phase 1 Checkpoint to Google Drive
# Run this cell after Phase 1 so you NEVER lose your trained weights!
# Tomorrow you can resume straight into Phase 2 without retraining!
# ==============================================================================
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    print('[INFO] Mounting Google Drive to save weights permanently...')
    drive.mount('/content/drive')

    DRIVE_DIR = '/content/drive/MyDrive/food-classification-dl/mobilenetv2'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Backup weights and history
    !cp results/mobilenetv2/best_phase1.keras "{DRIVE_DIR}/"
    !cp results/mobilenetv2/history_phase1.csv "{DRIVE_DIR}/"
    !cp results/mobilenetv2/model_summary.txt "{DRIVE_DIR}/"

    print(f'[SUCCESS] Phase 1 weights safely backed up to Google Drive at:')
    print(f'          {DRIVE_DIR}')
else:
    print('[INFO] Local environment detected. Weights are already in results/mobilenetv2/')


[INFO] Mounting Google Drive to save weights permanently...
Mounted at /content/drive
[SUCCESS] Phase 1 weights safely backed up to Google Drive at:
          /content/drive/MyDrive/food-classification-dl/mobilenetv2


In [4]:
# ==============================================================================
# Step 6: Configure & Train Phase 2 (Fine-Tuning Top Residual Blocks)
# Supports running continuously OR resuming tomorrow from saved checkpoint!
# ==============================================================================
import os
import sys
import time
import pandas as pd
import tensorflow as tf
from tensorflow.keras import callbacks
from src.models.mobilenetv2 import build_mobilenetv2_model, setup_fine_tuning

# 1. Restore model if restarting session tomorrow
phase1_checkpoint = 'results/mobilenetv2/best_phase1.keras'
drive_backup = '/content/drive/MyDrive/food-classification-dl/mobilenetv2/best_phase1.keras'

if 'model' not in locals():
    print('[INFO] Resuming session: restoring model from checkpoint...')
    if not os.path.exists(phase1_checkpoint) and os.path.exists(drive_backup):
        os.makedirs('results/mobilenetv2', exist_ok=True)
        !cp "{drive_backup}" "results/mobilenetv2/best_phase1.keras"
        !cp "/content/drive/MyDrive/food-classification-dl/mobilenetv2/history_phase1.csv" "results/mobilenetv2/"

    if os.path.exists(phase1_checkpoint):
        print(f'[INFO] Loading Phase 1 weights from: {phase1_checkpoint}')
        model = tf.keras.models.load_model(phase1_checkpoint)
    else:
        print('[WARNING] Checkpoint not found. Instantiating fresh model.')
        model = build_mobilenetv2_model(num_classes=101, trainable_base=False)

# Determine starting epoch
if 'history_phase1' in locals():
    initial_epoch = len(history_phase1.epoch)
elif os.path.exists('results/mobilenetv2/history_phase1.csv'):
    h1_df = pd.read_csv('results/mobilenetv2/history_phase1.csv')
    initial_epoch = len(h1_df)
else:
    initial_epoch = 8

# 2. Unfreeze top inverted residual blocks from layer 120 onward (lr=1e-5)
model = setup_fine_tuning(model, fine_tune_at_layer=120, learning_rate=1e-5)

PHASE2_EPOCHS = 12
TOTAL_EPOCHS = initial_epoch + PHASE2_EPOCHS

cb_phase2 = [
    callbacks.ModelCheckpoint(
        'results/mobilenetv2/best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print(f'[INFO] Starting Phase 2 fine-tuning from epoch {initial_epoch} to {TOTAL_EPOCHS}...')
start_time_phase2 = time.time()
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=initial_epoch,
    callbacks=cb_phase2
)
duration_phase2 = time.time() - start_time_phase2
print(f'[INFO] Phase 2 finished in {duration_phase2:.1f}s ({duration_phase2/60:.1f} min).')


[INFO] Resuming session: restoring model from checkpoint...
[INFO] Loading Phase 1 weights from: results/mobilenetv2/best_phase1.keras
[INFO] MobileNetV2 fine-tuning enabled from layer 120/154. Trainable base layers: 22.
[INFO] Starting Phase 2 fine-tuning from epoch 8 to 20...
Epoch 9/20
2131/2131 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step - accuracy: 0.6041 - loss: 1.5014 - top_5_accuracy: 0.8515
Epoch 9: val_loss improved from None to 1.66409, saving model to results/mobilenetv2/best_model.keras

Epoch 9: finished saving model to results/mobilenetv2/best_model.keras
2131/2131 ━━━━━━━━━━━━━━━━━━━━ 811s 369ms/step - accuracy: 0.6174 - loss: 1.4449 - top_5_accuracy: 0.8589 - val_accuracy: 0.5807 - val_loss: 1.6641 - val_top_5_accuracy: 0.8298 - learning_rate: 1.0000e-05
Epoch 10/20
2131/2131 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step - accuracy: 0.6307 - loss: 1.3928 - top_5_accuracy: 0.8677
Epoch 10: val_loss improved from 1.66409 to 1.57393, saving model to results/mobilenetv2/best_model.keras

Epoc

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/food-classification-dl/mobilenetv2"
os.makedirs(DRIVE_DIR, exist_ok=True)
!cp -r results/mobilenetv2/* "{DRIVE_DIR}/"
print("[SUCCESS] All files copied to Google Drive!")


In [3]:
# ==============================================================================
# Step 7: Export Combined Learning Curves & Training History
# ==============================================================================
import os
import pandas as pd
import matplotlib.pyplot as plt

# Load Phase 1 history from memory or saved CSV
if 'history_phase1' in locals():
    h1_dict = history_phase1.history
elif os.path.exists('results/mobilenetv2/history_phase1.csv'):
    h1_dict = pd.read_csv('results/mobilenetv2/history_phase1.csv', index_col='epoch').to_dict(orient='list')
else:
    h1_dict = {}

h2_dict = history_phase2.history
combined = {}
for k in h2_dict.keys():
    combined[k] = h1_dict.get(k, []) + h2_dict[k]

history_df = pd.DataFrame(combined)
history_df.index.name = 'epoch'
history_df.to_csv('results/mobilenetv2/history.csv')
print('[INFO] Saved results/mobilenetv2/history.csv')

# Plot High-Res Dual Learning Curves
split_epoch = len(h1_dict.get('loss', []))
plt.figure(figsize=(14, 5))

# 1. Loss Curve
plt.subplot(1, 2, 1)
plt.plot(history_df['loss'], label='Training Loss', color='#1f77b4', lw=2)
plt.plot(history_df['val_loss'], label='Validation Loss', color='#ff7f0e', lw=2)
if split_epoch > 0:
    plt.axvline(x=split_epoch - 1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('MobileNetV2 — Cross-Entropy Loss Curve', fontsize=13, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)

# 2. Top-1 Accuracy Curve
plt.subplot(1, 2, 2)
plt.plot(history_df['accuracy'], label='Training Top-1 Acc', color='#2ca02c', lw=2)
plt.plot(history_df['val_accuracy'], label='Validation Top-1 Acc', color='#d62728', lw=2)
if split_epoch > 0:
    plt.axvline(x=split_epoch - 1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('MobileNetV2 — Top-1 Accuracy Curve', fontsize=13, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Accuracy', fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(frameon=True)

plt.tight_layout()
plt.savefig('results/mobilenetv2/training_curves.png', dpi=300)
plt.show()
print('[INFO] Saved results/mobilenetv2/training_curves.png')


NameError: name 'history_phase2' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================================================================
# Step 8: Comprehensive Evaluation on Unseen Test Dataset (25,250 images)
# ==============================================================================
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import json

print("[INFO] Evaluating on Validation Set...")
val_loss, val_top1, val_top5 = model.evaluate(val_ds, verbose=1)

print("[INFO] Evaluating on Final Unseen Test Set (25,250 images)...")
test_loss, test_top1, test_top5 = model.evaluate(test_ds, verbose=1)

# Collect all true labels and predictions on test set
print("[INFO] Generating test predictions for classification report and confusion matrix...")
y_true = []
y_pred = []
for imgs, lbls in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(lbls.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Load class names
with open("data/splits/classes.txt") as f:
    class_names = [l.strip() for l in f if l.strip()]

# Compute detailed report
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
with open("results/mobilenetv2/classification_report.json", "w") as f:
    json.dump(report_dict, f, indent=2)
print("[INFO] Saved results/mobilenetv2/classification_report.json")


In [ ]:
# ==============================================================================
# Step 9: Plot & Save Normalized 101-Class Confusion Matrix
# ==============================================================================
import seaborn as sns

cm = confusion_matrix(y_true, y_pred, normalize="true")

plt.figure(figsize=(24, 22))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=False,
    yticklabels=False,
    cbar_kws={"shrink": 0.8}
)
plt.title("MobileNetV2 Normalized Confusion Matrix (101 Food Classes)", fontsize=18, fontweight="bold", pad=20)
plt.xlabel("Predicted Class Index (0-100)", fontsize=14)
plt.ylabel("True Class Index (0-100)", fontsize=14)
plt.tight_layout()
plt.savefig("results/mobilenetv2/confusion_matrix.png", dpi=300)
plt.show()
print("[INFO] Saved results/mobilenetv2/confusion_matrix.png")


In [ ]:
# ==============================================================================
# Step 10: Edge Latency Profiling (ms per Image Benchmark)
# ==============================================================================
print("[INFO] Benchmarking single-image inference latency across 1,000 samples...")
latencies = []

# Warm-up GPU/CPU
dummy_input = tf.random.normal([1, 224, 224, 3])
for _ in range(20):
    _ = model(dummy_input, training=False)

# Take 1,000 individual samples
count = 0
for imgs, _ in test_ds.unbatch().take(1000):
    single_img = tf.expand_dims(imgs, axis=0)
    t0 = time.perf_counter()
    _ = model(single_img, training=False)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0) # Convert to ms
    count += 1

avg_latency = float(np.mean(latencies))
std_latency = float(np.std(latencies))
p95_latency = float(np.percentile(latencies, 95))

print(f"\n--- MobileNetV2 Inference Latency Profile ---")
print(f"Average Latency: {avg_latency:.2f} ms/image")
print(f"Std Dev:         {std_latency:.2f} ms")
print(f"95th Percentile: {p95_latency:.2f} ms")


In [ ]:
# ==============================================================================
# Step 11: Export Standardized metrics.json and config.yaml
# ==============================================================================
import yaml

# Model size on disk
model_size_mb = round((model.count_params() * 4) / (1024 * 1024), 2)

metrics_dict = {
    "model_name": "MobileNetV2",
    "total_parameters": int(model.count_params()),
    "trainable_parameters": int(sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)),
    "model_size_mb": model_size_mb,
    "training_time_seconds": round(total_training_time, 2),
    "avg_epoch_time_seconds": round(total_training_time / len(history_df), 2),
    "inference_latency_ms_per_image": round(avg_latency, 2),
    "val_top1_accuracy": round(float(val_top1), 4),
    "val_top5_accuracy": round(float(val_top5), 4),
    "test_top1_accuracy": round(float(test_top1), 4),
    "test_top5_accuracy": round(float(test_top5), 4),
    "test_macro_precision": round(float(report_dict["macro avg"]["precision"]), 4),
    "test_macro_recall": round(float(report_dict["macro avg"]["recall"]), 4),
    "test_macro_f1": round(float(report_dict["macro avg"]["f1-score"]), 4)
}

with open("results/mobilenetv2/metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)
print("[INFO] Saved results/mobilenetv2/metrics.json")

# Snapshot config
mobilenet_config = {
    "model": "MobileNetV2",
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "phase1_epochs": len(history_phase1.epoch),
    "phase2_epochs": len(history_phase2.epoch),
    "phase1_optimizer": "Adam(lr=0.001)",
    "phase2_optimizer": "Adam(lr=1e-05)",
    "fine_tune_at_layer": 120,
    "loss": "sparse_categorical_crossentropy"
}

with open("results/mobilenetv2/config.yaml", "w") as f:
    yaml.dump(mobilenet_config, f, default_flow_style=False)
print("[INFO] Saved results/mobilenetv2/config.yaml")

print("\n[SUCCESS] ALL 7 STANDARDIZED ARTIFACTS EXPORTED TO results/mobilenetv2/")
